# 6. Delta Sharing – Compartir datos de Golden

Objetivo: demostrar el intercambio seguro de tablas analíticas de Unity Catalog con un destinatario externo.

Proveedor: `dbw_proyecto_final_ehs` · Catálogo: `dbw_proyecto_final_ehs` · Esquema: `golden`.

**Nota:** Delta Sharing es una demostración adicional, no un requisito obligatorio del trabajo final. Antes de ejecutar, confirma que OpenSharing está habilitado y que tu identidad tiene los privilegios requeridos en el metastore.

## 1. Conceptos

- **Provider:** nuestro metastore, que comparte los datos.
- **Share:** conjunto de tablas que se exponen.
- **Recipient:** identidad externa que recibe acceso.
- El destinatario recibe solo lectura (`SELECT`) sobre el share; no obtiene escritura sobre las tablas originales.

## 2. Verificar las tablas Golden

In [ ]:
SHOW TABLES IN dbw_proyecto_final_ehs.golden;

SELECT COUNT(*) AS total_ventas FROM dbw_proyecto_final_ehs.golden.fact_ventas;
SELECT COUNT(*) AS total_inventario FROM dbw_proyecto_final_ehs.golden.fact_inventario;

## 3. Crear el Share

Se compartirán tablas Golden; no se expondrán Bronze ni Silver. Ejecutar una sola vez y revisar el share si ya existe.

In [ ]:
CREATE SHARE IF NOT EXISTS share_golden_proyecto_final
COMMENT 'Share de demostracion del modelo analitico Golden de ventas e inventario';

## 4. Agregar tablas al Share

Se incluyen las dos tablas de hechos y sus dimensiones para permitir interpretar las claves.

In [ ]:
ALTER SHARE share_golden_proyecto_final ADD TABLE dbw_proyecto_final_ehs.golden.fact_ventas;
ALTER SHARE share_golden_proyecto_final ADD TABLE dbw_proyecto_final_ehs.golden.fact_inventario;
ALTER SHARE share_golden_proyecto_final ADD TABLE dbw_proyecto_final_ehs.golden.dim_fecha;
ALTER SHARE share_golden_proyecto_final ADD TABLE dbw_proyecto_final_ehs.golden.dim_cliente;
ALTER SHARE share_golden_proyecto_final ADD TABLE dbw_proyecto_final_ehs.golden.dim_producto;
ALTER SHARE share_golden_proyecto_final ADD TABLE dbw_proyecto_final_ehs.golden.dim_almacen;

## 5. Verificar contenido

In [ ]:
DESCRIBE SHARE share_golden_proyecto_final;

## 6. Crear destinatario de demostración (TOKEN / Open Sharing)

Esta modalidad sirve para un consumidor que no tiene otro metastore Databricks. Databricks genera un enlace de activación/credencial.

Nombre propuesto: `recipient_demo_golden`.

**No ejecutes esta celda si el objetivo es Databricks-to-Databricks.** Para ese caso necesitas el Sharing Identifier del metastore receptor y debes usar `CREATE RECIPIENT ... USING ID '<sharing-identifier>'`.

El enlace de activación es una credencial sensible: no publicarlo en GitHub, README ni capturas.

In [ ]:
CREATE RECIPIENT IF NOT EXISTS recipient_demo_golden
COMMENT 'Destinatario de demostracion para el proyecto final';

## 7. Conceder acceso de solo lectura

In [ ]:
GRANT SELECT ON SHARE share_golden_proyecto_final
TO RECIPIENT recipient_demo_golden;

## 8. Verificar destinatario y permisos

`DESCRIBE RECIPIENT` puede devolver `activation_link`. Tratarlo como secreto.

In [ ]:
DESCRIBE RECIPIENT recipient_demo_golden;
SHOW GRANTS ON SHARE share_golden_proyecto_final;
SHOW GRANTS TO RECIPIENT recipient_demo_golden;

## 9. Pasos para completar la demostración

1. Confirmar que el share contiene las seis tablas Golden.
2. Confirmar que el destinatario tiene únicamente `SELECT`.
3. Obtener el enlace de activación mediante `DESCRIBE RECIPIENT` o Catalog Explorer y entregarlo al consumidor por un canal privado.
4. El consumidor activa el acceso y utiliza el archivo de credenciales Delta Sharing con un cliente compatible.
5. No guardar el enlace ni el archivo de credenciales en el repositorio público.

La creación de shares y recipients requiere privilegios de Unity Catalog y OpenSharing habilitado. Si aparece un error de permisos, no eliminar ni recrear el metastore: revisar los privilegios `CREATE SHARE`, `CREATE RECIPIENT`, `USE SHARE`, `USE RECIPIENT` y la condición de Metastore Admin.

## 10. Documentación oficial

- Crear shares: https://learn.microsoft.com/azure/databricks/delta-sharing/create-share
- Crear recipients TOKEN: https://learn.microsoft.com/azure/databricks/delta-sharing/create-recipient-token
- Conceder acceso: https://learn.microsoft.com/azure/databricks/delta-sharing/grant-access
- Configurar OpenSharing: https://learn.microsoft.com/azure/databricks/delta-sharing/set-up